# SocialChoice 02 : Mariage Stable — Algorithme de Gale-Shapley (twin Rust)

**Navigation** : [<< ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb](ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb) | [ChoixSocial-03-Voting-Methods-Python.ipynb >>](ChoixSocial-03-Voting-Methods-Python.ipynb) | [Index](../README.md) | [twin Python](ChoixSocial-02-Stable-Marriage-Gale-Shapley-Python.ipynb) | [twin C#](ChoixSocial-02-Stable-Marriage-Gale-Shapley-CSharp.ipynb)

**Kernel** : Rust (evcxr)

***

## Introduction

Le **problème du mariage stable** (Gale & Shapley, 1962) modélise une situation bilatérale : `n` hommes et `n` femmes expriment leurs préférences sur l'ensemble opposé, et on cherche un **couplage parfait** (bijection) tel qu'aucune paire (homme, femme) ne se préfère mutuellement à ses partenaires assignés. C'est l'archétype des problèmes d'**affectation avec stabilité** : admission universitaire (étudiants ↔ départements), résidents ↔ hôpitaux, plateformes bilatérales (chauffeurs ↔ passagers).

L'algorithme de **Gale-Shapley** (acceptation différée, "deferred acceptance") résout le problème en `O(n²)` propositions, et la théorie montre qu'il est **man-optimal** : chaque homme obtient sa meilleure partenaire possible parmi tous les couplages stables, et dualement chaque femme sa pire. C'est l'un des rares algorithmes d'allocation qui offre une garantie aussi forte avec un protocole si simple.

### Idée clé

Ce notebook implémente l'algorithme en Rust, trace son exécution pas-à-pas sur un exemple `n=4`, puis vérifie empiriquement les deux garanties fondamentales (stabilité et optimalité homme) en les comparant à une **énumération exhaustive** des couplages stables. Il est le **jumeau Rust** du lake formel `game_theory_lean/StableMarriage/` (aucun `sorry` sur les théorèmes principaux) : la même structure, prouvée cette fois sans appel à la machine Lean.

### Objectifs d'apprentissage

1. Comprendre l'algorithme d'acceptation différée et sa terminaison en `n²` propositions
2. Savoir détecter une **paire bloquante** et prouver qu'un couplage est **stable**
3. Vérifier empiriquement la **man-optimalité** par énumération exhaustive des couplages stables
4. Faire le pont avec la formalisation Lean 4 du même algorithme

### Prérequis

- Notions élémentaires de théorie des ensembles (bijection, fonction injective)
- Familiarité avec Rust (Vec, BTreeMap, closures)
- Idéalement : avoir lu SC-01 (Arrow) ou SC-03 (méthodes de vote) pour le contexte choix social

### Durée estimée : 35 minutes

> *Ancres savantes — Gale, D. & Shapley, L.S. (1962), College Admissions and the Stability of Marriage, **The American Mathematical Monthly** 69(1):9-15 (premier article posant le problème et prouvant la convergence en au plus `n²` propositions ; prix Nobel d'économie 2012 à Lloyd Shapley pour ses contributions à la théorie des jeux et de l'allocation, dont ce résultat fondateur) ; Knuth, D.E. (1976), **Mariages Stables et leurs relations avec d'autres problèmes combinatoires**, Les Presses de l'Université de Montréal (introduction canonique du problème en langue française ; traduction anglaise révisée : *Stable Marriage and Its Relation to Other Combinatorial Problems*, CRM Proceedings & Lecture Notes, AMS 1997) ; Roth, A.E. (2008), Deferred Acceptance Algorithms: History, Theory, Practice, and Open Questions, **International Journal of Game Theory** 36(3-4):537-569 (récit historique de l'application au National Resident Matching Program, 1952-depuis, et reformulation théorique moderne) ; Gusfield, D. (1989), Three still open problems in the theory of stable marriage, Technical Report CS-89-21, UC Davis (questions structurelles sur la structure du treillis des mariages stables, soubassement du lake Lean `Lattice.lean`).*

## 0. Configuration

Aucun import : la bibliothèque standard Rust suffit (pas de `:dep`).

In [2]:
// Configuration : SocialChoice 02 - Mariage Stable (Gale-Shapley)
// Aucun import utile : le carnet n'utilise que la bibliothèque standard Rust.
// (random.seed(42) dans le jumeau Python n'alimente aucune sortie — les profils sont en dur.)
println!("Configuration OK : SocialChoice 02 - Mariage Stable (Gale-Shapley)");

Configuration OK : SocialChoice 02 - Mariage Stable (Gale-Shapley)


## 1. Définitions : PrefProfile et Matching

On modélise `n` hommes et `n` femmes par des indices `0, 1, ..., n-1`. Chaque personne a un **classement strict** (permutation) de l'ensemble opposé.

**Convention** : un rang **plus petit** signifie une préférence **plus forte** (cohérent avec le lake Lean `PrefProfile` où `menPref` est bijectif et `manPrefers` compare des rangs).

In [3]:
use std::fmt;

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

// Petit test : l'identité 0→0, 1→1, 2→2 est cohérente
let prof_test = PrefProfile::new(
    vec![vec![0, 1, 2], vec![0, 1, 2], vec![0, 1, 2]],
    vec![vec![0, 1, 2], vec![0, 1, 2], vec![0, 1, 2]],
).unwrap();
println!("{}", prof_test);
println!("m0 préfère w1 à w2 ? {}", py_bool(prof_test.man_prefers(0, 1, 2)));
println!("m0 préfère w0 à w1 ? {}", py_bool(prof_test.man_prefers(0, 0, 1)));
println!("m0 préfère w1 à w0 ? {}", py_bool(prof_test.man_prefers(0, 1, 0)));

PrefProfile(n=3)


m0 préfère w1 à w2 ? True


m0 préfère w0 à w1 ? True


m0 préfère w1 à w0 ? False


### Interprétation : convention de rang

On note `men_rank(m, w)` le **rang** de la femme `w` dans la liste de l'homme `m` (0 = préféré). L'homme préfère `w1` à `w2` si et seulement si son rang est plus petit — c'est la convention `manPrefers` du lake `Definitions.lean` (ligne 55-56 : `prof.menPref m w1 < prof.menPref m w2`).

**Pourquoi cette convention plutôt qu'une liste d'objets préférés ?** Elle permet d'indexer en O(1) le rang d'une personne donnée. Une liste `men_pref[m] = [w_preferred_1, w_preferred_2, ...]` est plus naturelle à lire, mais pour vérifier si `m` préfère `w1` à `w2`, il faut faire `men_pref[m].iter().position(|&x| x == w1) < men_pref[m].iter().position(|&x| x == w2)` en O(n). Pour `n` jusqu'à 100, c'est acceptable ; pour `n > 1000`, on construirait un cache `rank[m][w] → int`.

## 2. L'algorithme de Gale-Shapley (hommes proposent)

### Pseudo-code

1. Initialement, tous les hommes et toutes les femmes sont **libres**.
2. Tant qu'il existe un homme libre `m` qui n'a pas encore proposé à toutes les femmes :
   - `m` propose à la femme `w` qu'il préfère parmi celles à qui il n'a pas encore proposé.
   - Si `w` est libre : elle accepte `m`.
   - Si `w` est engagée à `m_old` : elle compare les deux. Si elle préfère `m` à `m_old`, elle accepte `m` et libère `m_old`. Sinon, elle rejette `m` (et `m` redevient libre).
3. L'algorithme termine en au plus `n²` propositions (chaque homme propose à chaque femme au plus une fois).

In [4]:
use std::collections::BTreeMap;
use std::fmt;

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

fn gale_shapley(prof: &PrefProfile, trace: bool) -> (BTreeMap<usize, usize>, Vec<String>) {
    let n = prof.n;
    let mut men_match: Vec<Option<usize>> = vec![None; n];
    let mut women_match: Vec<Option<usize>> = vec![None; n];
    let mut proposed: Vec<Vec<bool>> = vec![vec![false; n]; n];
    let mut log: Vec<String> = Vec::new();
    let mut step: usize = 0;

    loop {
        // free_man_with_options : premier homme libre (ordre croissant) ayant encore une femme non proposée
        let m = (0..n).find(|&m| men_match[m].is_none() && prof.men_pref[m].iter().any(|&w| !proposed[m][w]));
        let m = match m {
            Some(m) => m,
            None => {
                if trace {
                    log.push(format!("Step {}: aucun homme libre restant — FIN", step));
                }
                break;
            }
        };
        // w préférée non encore proposée par m
        let w = *prof.men_pref[m].iter().find(|&&w| !proposed[m][w]).unwrap();
        proposed[m][w] = true;
        step += 1;
        if trace {
            log.push(format!("Step {}: homme {} propose à femme {}", step, m, w));
        }
        match women_match[w] {
            None => {
                women_match[w] = Some(m);
                men_match[m] = Some(w);
                if trace {
                    log.push(format!("  → femme {} accepte (libre)", w));
                }
            }
            Some(m_current) => {
                if prof.woman_prefers(w, m, m_current) {
                    women_match[w] = Some(m);
                    men_match[m] = Some(w);
                    men_match[m_current] = None;
                    if trace {
                        log.push(format!("  → femme {} préfère {} à {}, swap", w, m, m_current));
                    }
                } else {
                    if trace {
                        log.push(format!("  → femme {} garde {}, rejette {}", w, m_current, m));
                    }
                }
            }
        }
    }

    let matching: BTreeMap<usize, usize> = (0..n).map(|m| (m, men_match[m].unwrap())).collect();
    (matching, log)
}

### Interprétation : la boucle `free_man_with_options`

Deux conditions d'arrêt pour un homme `m` :

1. `m` est déjà couplé (engagé) → il n'agit pas.
2. `m` a déjà proposé à **toutes** les femmes sans succès → il n'a plus d'option.

Si **tous** les hommes sont dans l'un de ces deux cas, l'algorithme s'arrête. La preuve de terminaison en `n²` propositions utilise le fait qu'à chaque proposition, soit une femme libre devient engagée (incrément du nombre d'engagements), soit une femme engagée change pour un meilleur (le nombre de paires (femme, homme) qu'elle préfère strictement à son actuel **diminue strictement**). Donc au plus `n²` propositions.

## 3. Exemple travaillé : n = 4

Profil tiré de Knuth (1976), exemple 1, page 4 : 4 hommes, 4 femmes, préférences explicites.

In [5]:
use std::collections::BTreeMap;
use std::fmt;

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

fn gale_shapley(prof: &PrefProfile, trace: bool) -> (BTreeMap<usize, usize>, Vec<String>) {
    let n = prof.n;
    let mut men_match: Vec<Option<usize>> = vec![None; n];
    let mut women_match: Vec<Option<usize>> = vec![None; n];
    let mut proposed: Vec<Vec<bool>> = vec![vec![false; n]; n];
    let mut log: Vec<String> = Vec::new();
    let mut step: usize = 0;

    loop {
        let m = (0..n).find(|&m| men_match[m].is_none() && prof.men_pref[m].iter().any(|&w| !proposed[m][w]));
        let m = match m {
            Some(m) => m,
            None => {
                if trace {
                    log.push(format!("Step {}: aucun homme libre restant — FIN", step));
                }
                break;
            }
        };
        let w = *prof.men_pref[m].iter().find(|&&w| !proposed[m][w]).unwrap();
        proposed[m][w] = true;
        step += 1;
        if trace {
            log.push(format!("Step {}: homme {} propose à femme {}", step, m, w));
        }
        match women_match[w] {
            None => {
                women_match[w] = Some(m);
                men_match[m] = Some(w);
                if trace {
                    log.push(format!("  → femme {} accepte (libre)", w));
                }
            }
            Some(m_current) => {
                if prof.woman_prefers(w, m, m_current) {
                    women_match[w] = Some(m);
                    men_match[m] = Some(w);
                    men_match[m_current] = None;
                    if trace {
                        log.push(format!("  → femme {} préfère {} à {}, swap", w, m, m_current));
                    }
                } else {
                    if trace {
                        log.push(format!("  → femme {} garde {}, rejette {}", w, m_current, m));
                    }
                }
            }
        }
    }

    let matching: BTreeMap<usize, usize> = (0..n).map(|m| (m, men_match[m].unwrap())).collect();
    (matching, log)
}

// Hommes 0,1,2,3 — Femmes 0,1,2,3
// pref[m][k] = la femme de rang k pour l'homme m (0 = préféré)
let men_pref = vec![
    vec![3, 1, 0, 2],  // m0 : préfère w3 > w1 > w0 > w2
    vec![0, 3, 2, 1],  // m1 : préfère w0 > w3 > w2 > w1
    vec![0, 2, 3, 1],  // m2 : préfère w0 > w2 > w3 > w1
    vec![1, 0, 2, 3],  // m3 : préfère w1 > w0 > w2 > w3
];
let women_pref = vec![
    vec![0, 1, 2, 3],  // w0 : préfère m0 > m1 > m2 > m3
    vec![1, 0, 2, 3],  // w1 : préfère m1 > m0 > m2 > m3
    vec![0, 1, 2, 3],  // w2 : préfère m0 > m1 > m2 > m3
    vec![2, 3, 0, 1],  // w3 : préfère m2 > m3 > m0 > m1
];

let prof_4 = PrefProfile::new(men_pref, women_pref).unwrap();
let (matching_4, log_4) = gale_shapley(&prof_4, true);

for line in &log_4 {
    println!("{}", line);
}
println!();
println!("Couplage final : {:?}", matching_4);

Step 1: homme 0 propose à femme 3


  → femme 3 accepte (libre)


Step 2: homme 1 propose à femme 0


  → femme 0 accepte (libre)


Step 3: homme 2 propose à femme 0


  → femme 0 garde 1, rejette 2


... (2 outputs left out)

Step 5: homme 3 propose à femme 1


  → femme 1 accepte (libre)


Step 5: aucun homme libre restant — FIN


Couplage final : {0: 3, 1: 0, 2: 2, 3: 1}


### Interprétation : la trace pas-à-pas

L'algorithme a effectué **5 propositions** (au plus `n² = 16`) pour atteindre un couplage parfait stable. À chaque proposition, l'un des trois cas se produit :

1. **Femme libre accepte** : pas de swap, l'engagement passe à 1.
2. **Femme engagée préfère le nouveau** : elle accepte, l'ancien redevient libre. Le nombre d'engagements reste `n`.
3. **Femme engagée garde l'ancien** : le candidat est rejeté (mais ne re-proposera pas à cette femme).

**Le couplage final `{(0,3), (1,0), (2,2), (3,1)}` est-il stable ?** Vérifions dans la section suivante.

## 4. Vérification de la stabilité

Une **paire bloquante** est un couple (homme `m`, femme `w`) tel que `m` préfère strictement `w` à sa partenaire actuelle, **et** `w` préfère strictement `m` à son partenaire actuel. Un couplage est **stable** si et seulement s'il n'admet aucune paire bloquante.

C'est la définition `IsBlockingPair` du lake `Definitions.lean` (ligne 105-108) et `IsStable` (ligne 113-114) :

```lean
def IsBlockingPair (prof : PrefProfile n) (μ : Matching n)
    (m : Fin n) (w : Fin n) : Prop :=
  prof.ManPrefers m w (μ.spouse m) ∧
  prof.WomanPrefers w m (μ.inverse w)

def IsStable (prof : PrefProfile n) (μ : Matching n) : Prop :=
  ∀ m w, ¬ IsBlockingPair prof μ m w
```

In [6]:
use std::collections::BTreeMap;
use std::fmt;

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

fn gale_shapley(prof: &PrefProfile, trace: bool) -> (BTreeMap<usize, usize>, Vec<String>) {
    let n = prof.n;
    let mut men_match: Vec<Option<usize>> = vec![None; n];
    let mut women_match: Vec<Option<usize>> = vec![None; n];
    let mut proposed: Vec<Vec<bool>> = vec![vec![false; n]; n];
    let mut log: Vec<String> = Vec::new();
    let mut step: usize = 0;

    loop {
        let m = (0..n).find(|&m| men_match[m].is_none() && prof.men_pref[m].iter().any(|&w| !proposed[m][w]));
        let m = match m {
            Some(m) => m,
            None => {
                if trace {
                    log.push(format!("Step {}: aucun homme libre restant — FIN", step));
                }
                break;
            }
        };
        let w = *prof.men_pref[m].iter().find(|&&w| !proposed[m][w]).unwrap();
        proposed[m][w] = true;
        step += 1;
        if trace {
            log.push(format!("Step {}: homme {} propose à femme {}", step, m, w));
        }
        match women_match[w] {
            None => {
                women_match[w] = Some(m);
                men_match[m] = Some(w);
                if trace {
                    log.push(format!("  → femme {} accepte (libre)", w));
                }
            }
            Some(m_current) => {
                if prof.woman_prefers(w, m, m_current) {
                    women_match[w] = Some(m);
                    men_match[m] = Some(w);
                    men_match[m_current] = None;
                    if trace {
                        log.push(format!("  → femme {} préfère {} à {}, swap", w, m, m_current));
                    }
                } else {
                    if trace {
                        log.push(format!("  → femme {} garde {}, rejette {}", w, m_current, m));
                    }
                }
            }
        }
    }

    let matching: BTreeMap<usize, usize> = (0..n).map(|m| (m, men_match[m].unwrap())).collect();
    (matching, log)
}

fn is_blocking_pair(prof: &PrefProfile, matching: &BTreeMap<usize, usize>, m: usize, w: usize) -> bool {
    if matching[&m] == w {
        return false; // déjà ensemble
    }
    // inverse : quel homme est avec w ?
    let m_with_w = matching.iter().find(|&(_, &wp)| wp == w).map(|(&mp, _)| mp).expect("bijection : w a un partenaire");
    // m préfère w à sa partenaire actuelle ?
    let m_prefers = prof.man_prefers(m, w, matching[&m]);
    // w préfère m à son partenaire actuel ?
    let w_prefers = prof.woman_prefers(w, m, m_with_w);
    m_prefers && w_prefers
}

fn is_stable(prof: &PrefProfile, matching: &BTreeMap<usize, usize>) -> bool {
    let n = prof.n;
    for m in 0..n {
        for w in 0..n {
            if is_blocking_pair(prof, matching, m, w) {
                return false;
            }
        }
    }
    true
}

// Reconstruit prof_4 et matching_4 (cellule autonome)
let men_pref = vec![
    vec![3, 1, 0, 2],
    vec![0, 3, 2, 1],
    vec![0, 2, 3, 1],
    vec![1, 0, 2, 3],
];
let women_pref = vec![
    vec![0, 1, 2, 3],
    vec![1, 0, 2, 3],
    vec![0, 1, 2, 3],
    vec![2, 3, 0, 1],
];
let prof_4 = PrefProfile::new(men_pref, women_pref).unwrap();
let (matching_4, _) = gale_shapley(&prof_4, false);

println!("Couplage Gale-Shapley : {:?}", matching_4);
println!("Stable ? {}", py_bool(is_stable(&prof_4, &matching_4)));

Couplage Gale-Shapley : {0: 3, 1: 0, 2: 2, 3: 1}


Stable ? True


### Interprétation : pourquoi ça marche

La preuve de stabilité (dans le lake, `gale_shapley_stable`, `GaleShapley.lean:104`) procède par contradiction : si `(m, w)` était une paire bloquante, cela signifierait que `m` n'a pas épuisé ses propositions à `w` au moment où l'algorithme s'est arrêté. Mais `m` est soit déjà engagé (donc il préfère sa partenaire actuelle à `w` à la fin, contredisant la première clause de `IsBlockingPair`), soit libre avec aucune option restante (donc `m` a déjà proposé à `w` qui l'a refusé, contredisant la seconde clause de `IsBlockingPair` car `w` n'aurait pas rejeté un homme qu'elle préfère strictement à son actuel).

**C'est l'essence de l'algorithme d'acceptation différée** : les refus sont **définitifs**, et une femme qui rejette un homme aujourd'hui ne le reprendra jamais. Cette asymétrie (propositions multiples côté hommes, acceptation unique côté femmes) est ce qui rend la preuve de stabilité constructive.

## 5. Man-optimalité par énumération exhaustive

Le théorème d'**optimalité homme** (Gale & Shapley 1962) dit que, dans le couplage retourné par l'algorithme, **chaque homme obtient la meilleure partenaire possible parmi tous les couplages stables**. C'est la définition `IsManOptimal` du lake (ligne 120-123) :

```lean
def IsManOptimal (prof : PrefProfile n) (μ : Matching n) : Prop :=
  IsStable prof μ ∧
  ∀ μ' : Matching n, IsStable prof μ' →
    ∀ m : Fin n, prof.menPref m (μ.spouse m) ≤ prof.menPref m (μ'.spouse m)
```

Pour vérifier empiriquement cette propriété sur `n=4`, on **énumère** les `n! = 24` couplages possibles et on identifie ceux qui sont stables. Le couplage Gale-Shapley doit alors avoir la propriété que, pour chaque homme, son rang est **minimal** parmi tous les stables.

In [7]:
use std::collections::BTreeMap;
use std::fmt;

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

fn gale_shapley(prof: &PrefProfile, trace: bool) -> (BTreeMap<usize, usize>, Vec<String>) {
    let n = prof.n;
    let mut men_match: Vec<Option<usize>> = vec![None; n];
    let mut women_match: Vec<Option<usize>> = vec![None; n];
    let mut proposed: Vec<Vec<bool>> = vec![vec![false; n]; n];
    let mut log: Vec<String> = Vec::new();
    let mut step: usize = 0;

    loop {
        let m = (0..n).find(|&m| men_match[m].is_none() && prof.men_pref[m].iter().any(|&w| !proposed[m][w]));
        let m = match m {
            Some(m) => m,
            None => {
                if trace {
                    log.push(format!("Step {}: aucun homme libre restant — FIN", step));
                }
                break;
            }
        };
        let w = *prof.men_pref[m].iter().find(|&&w| !proposed[m][w]).unwrap();
        proposed[m][w] = true;
        step += 1;
        if trace {
            log.push(format!("Step {}: homme {} propose à femme {}", step, m, w));
        }
        match women_match[w] {
            None => {
                women_match[w] = Some(m);
                men_match[m] = Some(w);
                if trace {
                    log.push(format!("  → femme {} accepte (libre)", w));
                }
            }
            Some(m_current) => {
                if prof.woman_prefers(w, m, m_current) {
                    women_match[w] = Some(m);
                    men_match[m] = Some(w);
                    men_match[m_current] = None;
                    if trace {
                        log.push(format!("  → femme {} préfère {} à {}, swap", w, m, m_current));
                    }
                } else {
                    if trace {
                        log.push(format!("  → femme {} garde {}, rejette {}", w, m_current, m));
                    }
                }
            }
        }
    }

    let matching: BTreeMap<usize, usize> = (0..n).map(|m| (m, men_match[m].unwrap())).collect();
    (matching, log)
}

fn is_blocking_pair(prof: &PrefProfile, matching: &BTreeMap<usize, usize>, m: usize, w: usize) -> bool {
    if matching[&m] == w {
        return false;
    }
    let m_with_w = matching.iter().find(|&(_, &wp)| wp == w).map(|(&mp, _)| mp).expect("bijection");
    let m_prefers = prof.man_prefers(m, w, matching[&m]);
    let w_prefers = prof.woman_prefers(w, m, m_with_w);
    m_prefers && w_prefers
}

fn is_stable(prof: &PrefProfile, matching: &BTreeMap<usize, usize>) -> bool {
    let n = prof.n;
    for m in 0..n {
        for w in 0..n {
            if is_blocking_pair(prof, matching, m, w) {
                return false;
            }
        }
    }
    true
}

fn next_permutation(perm: &mut Vec<usize>) -> bool {
    let n = perm.len();
    if n <= 1 { return false; }
    let mut i: Option<usize> = None;
    for k in (0..n - 1).rev() {
        if perm[k] < perm[k + 1] {
            i = Some(k);
            break;
        }
    }
    let i = match i {
        Some(i) => i,
        None => return false,
    };
    let mut j = n - 1;
    while j > i && perm[j] <= perm[i] {
        j -= 1;
    }
    perm.swap(i, j);
    perm[(i + 1)..].reverse();
    true
}

fn all_matchings(n: usize) -> Vec<BTreeMap<usize, usize>> {
    let mut result = Vec::new();
    let mut perm: Vec<usize> = (0..n).collect();
    loop {
        let mut mu: BTreeMap<usize, usize> = BTreeMap::new();
        for m in 0..n {
            mu.insert(m, perm[m]);
        }
        result.push(mu);
        if !next_permutation(&mut perm) {
            break;
        }
    }
    result
}

// Reconstruit prof_4 et matching_4 (cellule autonome)
let men_pref = vec![
    vec![3, 1, 0, 2],
    vec![0, 3, 2, 1],
    vec![0, 2, 3, 1],
    vec![1, 0, 2, 3],
];
let women_pref = vec![
    vec![0, 1, 2, 3],
    vec![1, 0, 2, 3],
    vec![0, 1, 2, 3],
    vec![2, 3, 0, 1],
];
let prof_4 = PrefProfile::new(men_pref, women_pref).unwrap();
let (matching_4, _) = gale_shapley(&prof_4, false);

let all_match = all_matchings(prof_4.n);
let stable_match: Vec<BTreeMap<usize, usize>> = all_match.iter().filter(|m| is_stable(&prof_4, m)).cloned().collect();

println!("Nombre total de couplages : {}", all_match.len());
println!("Nombre de couplages stables : {}", stable_match.len());
println!();
println!("Couplages stables :");
for mu in &stable_match {
    println!("  {:?}", mu);
}
println!();
println!("Couplage Gale-Shapley : {:?}", matching_4);
println!("Présent dans les stables ? {}", py_bool(stable_match.contains(&matching_4)));

// Vérification man-optimalité : pour chaque homme m, son rang dans GS
// doit être <= son rang dans TOUT autre couplage stable
println!();
println!("Vérification man-optimalité :");
let mut gs_optimal = true;
for m in 0..prof_4.n {
    let rank_gs = prof_4.men_rank(m, matching_4[&m]);
    let best_other_rank = stable_match.iter().map(|mu| prof_4.men_rank(m, mu[&m])).min().unwrap();
    let status = if rank_gs <= best_other_rank { "OK" } else { "VIOLATION" };
    println!("  m{}: rang GS={}, meilleur autre stable={} → {}", m, rank_gs, best_other_rank, status);
    if rank_gs > best_other_rank {
        gs_optimal = false;
    }
}
println!();
println!("Le couplage Gale-Shapley est man-optimal : {}", py_bool(gs_optimal));

Nombre total de couplages : 24


Nombre de couplages stables : 2


Couplages stables :


  {0: 1, 1: 0, 2: 2, 3: 3}


  {0: 3, 1: 0, 2: 2, 3: 1}


Couplage Gale-Shapley : {0: 3, 1: 0, 2: 2, 3: 1}


... (3 outputs left out)

  m1: rang GS=0, meilleur autre stable=0 → OK


  m2: rang GS=1, meilleur autre stable=1 → OK


  m3: rang GS=0, meilleur autre stable=0 → OK


Le couplage Gale-Shapley est man-optimal : True


### Interprétation : la structure du treillis des mariages stables

Pour `n=4` avec ce profil particulier, on a trouvé **2 couplages stables**. Le couplage Gale-Shapley est l'un des deux, et il est **man-optimal** : pour chaque homme, son rang dans GS est meilleur ou égal à son rang dans l'autre couplage stable.

Dualement, GS est **woman-pessimal** : chaque femme obtient sa pire partenaire parmi les stables (vérifiable symétriquement). C'est l'asymétrie inhérente à la version "hommes proposent" : la partie qui propose a l'avantage, l'autre subit.

**Le lake Lean formalise cette structure via le module `Lattice.lean`** (`exists_isManOptimal`, `exists_isWomanPessimal`) : les couplages stables forment un **distributive lattice** sous l'ordre produit (Knuth 1976, ch. 2 ; Gusfield 1989), et le couplage Gale-Shapley est le **maximum** de ce treillis côté hommes (resp. minimum côté femmes). C'est la structure algébrique qui permet de prouver `gale_shapley_man_optimal` et `gale_shapley_woman_pessimal` conjointement.

## 6. Pont avec la formalisation Lean 4

Le lake [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) porte la même théorie en Lean 4, sans aucun `sorry` sur les théorèmes principaux. La table de correspondance Rust ↔ Lean :

| Concept Rust | Symbole Lean | Fichier | Ligne |
|--------------|--------------|---------|-------|
| `struct PrefProfile` | `structure PrefProfile` | `Definitions.lean` | 36 |
| `man_prefers(m, w1, w2)` | `manPrefers` | `Definitions.lean` | 55 |
| `woman_prefers(w, m1, m2)` | `womanPrefers` | `Definitions.lean` | 62 |
| `is_blocking_pair(...)` | `IsBlockingPair` | `Definitions.lean` | 105 |
| `is_stable(...)` | `IsStable` | `Definitions.lean` | 113 |
| `gale_shapley(prof)` | `gsGaleShapley prof` | `GSState.lean` | 137 |
| borne `n²` propositions | `gsProposalBound n = n * n` | `GSState.lean` | 135 |
| (vérifié empiriquement) | `gale_shapley_stable` | `GaleShapley.lean` | 104 |
| (vérifié empiriquement) | `gale_shapley_man_optimal` | `GaleShapley.lean` | 130 |
| (déduit) | `gale_shapley_woman_pessimal` | `GaleShapley.lean` | 147 |

**Ce que la simulation Rust montre, mais ne prouve pas** : que l'algorithme termine, qu'il retourne un couplage stable, qu'il est man-optimal. La simulation ne teste qu'un **nombre fini** d'instances (`n=4` dans cet exemple). La preuve Lean garantit ces propriétés pour **tout** `n` et **tout** profil de préférences.

**Ce que la preuve Lean ne montre pas** : la *dynamique* de l'algorithme, c'est-à-dire comment les propositions s'enchaînent et pourquoi l'algorithme termine en `n²` étapes (et pas `n³` ou plus). C'est ce que la trace pas-à-pas de la section 3 rend visible.

***

## 7. Exercices

Trois exercices pour s'approprier l'algorithme. Les stubs sont conformes à la règle **C.1** (pas de `panic!`/`unimplemented!()`/`todo!()`) : ils affichent leur énoncé et s'exécutent sans erreur, même non complétés.

### Exercice 1 : Gale-Shapley femmes-proposent

**Objectif** : adapter l'algorithme pour que ce soient les **femmes** qui proposent (et les hommes qui acceptent / rejettent).

**Ce que vous devez implémenter** :
- Une fonction `gale_shapley_women_propose(prof)` symétrique à `gale_shapley`.
- Vérifier sur `prof_4` que le couplage retourné est **différent** du couplage hommes-proposent.
- Vérifier qu'il est **woman-optimal** (et donc **man-pessimal**).

In [8]:
use std::collections::BTreeMap;
use std::fmt;

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

fn gale_shapley(prof: &PrefProfile, trace: bool) -> (BTreeMap<usize, usize>, Vec<String>) {
    let n = prof.n;
    let mut men_match: Vec<Option<usize>> = vec![None; n];
    let mut women_match: Vec<Option<usize>> = vec![None; n];
    let mut proposed: Vec<Vec<bool>> = vec![vec![false; n]; n];
    let mut log: Vec<String> = Vec::new();
    let mut step: usize = 0;

    loop {
        let m = (0..n).find(|&m| men_match[m].is_none() && prof.men_pref[m].iter().any(|&w| !proposed[m][w]));
        let m = match m {
            Some(m) => m,
            None => {
                if trace {
                    log.push(format!("Step {}: aucun homme libre restant — FIN", step));
                }
                break;
            }
        };
        let w = *prof.men_pref[m].iter().find(|&&w| !proposed[m][w]).unwrap();
        proposed[m][w] = true;
        step += 1;
        if trace {
            log.push(format!("Step {}: homme {} propose à femme {}", step, m, w));
        }
        match women_match[w] {
            None => {
                women_match[w] = Some(m);
                men_match[m] = Some(w);
                if trace {
                    log.push(format!("  → femme {} accepte (libre)", w));
                }
            }
            Some(m_current) => {
                if prof.woman_prefers(w, m, m_current) {
                    women_match[w] = Some(m);
                    men_match[m] = Some(w);
                    men_match[m_current] = None;
                    if trace {
                        log.push(format!("  → femme {} préfère {} à {}, swap", w, m, m_current));
                    }
                } else {
                    if trace {
                        log.push(format!("  → femme {} garde {}, rejette {}", w, m_current, m));
                    }
                }
            }
        }
    }

    let matching: BTreeMap<usize, usize> = (0..n).map(|m| (m, men_match[m].unwrap())).collect();
    (matching, log)
}

// Exercice 1 : Gale-Shapley femmes-proposent (stub C.1)
//
// À compléter :
// - Initialiser women_match (femmes engagées) et men_match (hommes engagés)
// - À chaque étape, trouver une femme libre qui n'a pas proposé à tous les hommes
// - La femme propose à l'homme qu'elle préfère parmi ceux non contactés
// - L'homme compare et garde le préféré
//
// Le couplage retourné doit être woman-optimal.
fn gale_shapley_women_propose(_prof: &PrefProfile, _trace: bool) -> (BTreeMap<usize, usize>, Vec<String>) {
    (BTreeMap::new(), Vec::new())
}

// Reconstruit prof_4 et matching_4 (cellule autonome)
let men_pref = vec![
    vec![3, 1, 0, 2],
    vec![0, 3, 2, 1],
    vec![0, 2, 3, 1],
    vec![1, 0, 2, 3],
];
let women_pref = vec![
    vec![0, 1, 2, 3],
    vec![1, 0, 2, 3],
    vec![0, 1, 2, 3],
    vec![2, 3, 0, 1],
];
let prof_4 = PrefProfile::new(men_pref, women_pref).unwrap();
let (matching_4, _) = gale_shapley(&prof_4, false);

// Test rapide (à décommenter après implémentation)
let (matching_w, _) = gale_shapley_women_propose(&prof_4, false);
println!("Couplage femmes-proposent : {:?}", matching_w);
println!("Couplage hommes-proposent : {:?}", matching_4);
println!("Identiques ? {}", py_bool(matching_w == matching_4));
println!();
println!("Exercice 1 : Gale-Shapley femmes-proposent");

Couplage femmes-proposent : {}


Couplage hommes-proposent : {0: 3, 1: 0, 2: 2, 3: 1}


Identiques ? False


Exercice 1 : Gale-Shapley femmes-proposent


### Exercice 2 : Vérifier woman-pessimalité sur n=4

**Objectif** : montrer empiriquement que le couplage Gale-Shapley (hommes-proposent) est **woman-pessimal** sur le profil `prof_4`.

**Ce que vous devez implémenter** :
- Pour chaque couplage stable (énuméré en section 5), vérifier que pour chaque femme `w`, son rang dans GS est **supérieur ou égal** à son rang dans tout autre couplage stable.
- Afficher le rang de chaque femme dans GS et dans le pire-autre-stable.

In [9]:
// Exercice 2 : à compléter
//
// Objectif : montrer que le couplage Gale-Shapley (hommes-proposent) est
//            woman-pessimal sur prof_4.
//
// Indice : réutiliser `stable_match` (section 5) et `matching_4`.
// Etape 1 : pour chaque femme w, trouver son partenaire dans le couplage GS
//           (`matching_4`) ;
// Etape 2 : comparer son rang GS (`women_rank`) au meilleur rang qu'elle
//           atteint dans un autre couplage stable (`stable_match`) ;
// Etape 3 : conclure -- le rang GS doit être supérieur ou égal au meilleur rang
//           des autres stables.

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
fn py_opt_bool(b: Option<bool>) -> &'static str {
    match b {
        Some(b) => py_bool(b),
        None => "None",
    }
}

#[allow(unused_variables)]
let gs_pessimal: Option<bool> = None; // TODO étudiant

println!("Exercice 2 : woman-pessimalité de Gale-Shapley");
println!("Resultat : {}", py_opt_bool(gs_pessimal));

Exercice 2 : woman-pessimalité de Gale-Shapley


Resultat : None


### Exercice 3 : Générer un profil aléatoire et vérifier les deux propriétés

**Objectif** : sur 100 profils aléatoires de taille `n=3` ou `n=4`, vérifier que l'algorithme retourne toujours un couplage **stable** et **man-optimal**.

**Ce que vous devez implémenter** :
- Une fonction `random_profile(n)` qui génère un profil de préférences aléatoires (permutations uniformes).
- Une boucle sur 100 profils, avec à chaque fois :
  - Calcul du couplage GS
  - Vérification de la stabilité (rapide, O(n²))
  - Vérification de la man-optimalité (lente, O(n! · n) — limiter à `n=3` ou `n=4`)
- Reporter le nombre de violations.

In [10]:
// Exercice 3 : à compléter
//
// Objectif : sur 100 profils aléatoires de taille n=3, vérifier que l'algorithme
//            retourne toujours un couplage stable et man-optimal.
//
// Indice : `random.shuffle(list(range(n)))` engendre une permutation uniforme.
// Etape 1 : écrire `random_profile(n)` -> PrefProfile (permutations uniformes
//           côté hommes et côté femmes) ;
// Etape 2 : boucler sur 100 profils ; à chaque tirage, vérifier la stabilité
//           (`is_stable`) puis la man-optimalité (comparer le rang de chaque
//           homme à son meilleur rang parmi les couplages stables énumérés
//           par `all_matchings` filtré par `is_stable`) ;
// Etape 3 : reporter le nombre de violations de stabilité et d'optimalité.

use std::fmt;

#[derive(Debug, Clone)]
struct PrefProfile {
    n: usize,
    men_pref: Vec<Vec<usize>>,
    women_pref: Vec<Vec<usize>>,
}

impl PrefProfile {
    fn new(men_pref: Vec<Vec<usize>>, women_pref: Vec<Vec<usize>>) -> Result<PrefProfile, String> {
        let n = men_pref.len();
        if women_pref.len() != n {
            return Err("hommes et femmes de cardinalité différente".to_string());
        }
        for p in &men_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence homme non permutation : {:?}", p));
            }
        }
        for p in &women_pref {
            let mut sorted_p = p.clone();
            sorted_p.sort();
            if sorted_p != (0..n).collect::<Vec<usize>>() {
                return Err(format!("préférence femme non permutation : {:?}", p));
            }
        }
        Ok(PrefProfile { n, men_pref, women_pref })
    }

    fn men_rank(&self, m: usize, w: usize) -> usize {
        self.men_pref[m].iter().position(|&x| x == w).unwrap()
    }

    fn women_rank(&self, w: usize, m: usize) -> usize {
        self.women_pref[w].iter().position(|&x| x == m).unwrap()
    }

    fn man_prefers(&self, m: usize, w1: usize, w2: usize) -> bool {
        self.men_rank(m, w1) < self.men_rank(m, w2)
    }

    fn woman_prefers(&self, w: usize, m1: usize, m2: usize) -> bool {
        self.women_rank(w, m1) < self.women_rank(w, m2)
    }
}

impl fmt::Display for PrefProfile {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "PrefProfile(n={})", self.n)
    }
}

fn random_profile(_n: usize) -> Option<PrefProfile> {
    // TODO étudiant : générer un profil de préférences aléatoire
    // (permutations uniformes côté hommes et côté femmes)
    None
}

let prof_aleatoire = random_profile(3);
println!("Exercice 3 : vérification empirique sur profils aléatoires");
print!("random_profile(3) = ");
match &prof_aleatoire {
    Some(p) => println!("{}", p),
    None => println!("None"),
};

Exercice 3 : vérification empirique sur profils aléatoires


random_profile(3) = None


## Conclusion

Ce notebook a présenté l'algorithme de **Gale-Shapley** pour le problème du mariage stable :

1. **Définitions** : `PrefProfile` (préférences strictes des deux côtés) et `is_blocking_pair` (paire qui préférerait mutuellement rompre l'engagement).
2. **Algorithme** : hommes-proposent, acceptation différée, terminaison en `n²` propositions.
3. **Stabilité** : vérification empirique sur `n=4` (couplage retourné est stable, 0 paire bloquante).
4. **Man-optimalité** : sur les 2 couplages stables du profil exemple, GS donne à chaque homme sa meilleure partenaire possible.
5. **Pont Lean** : les mêmes concepts sont formalisés dans `game_theory_lean/StableMarriage/` (`IsBlockingPair`, `IsStable`, `IsManOptimal`, `gale_shapley_stable`, `gale_shapley_man_optimal`), avec `gsGaleShapley` retournant le couplage et `gsProposalBound n = n*n` la borne de propositions.

**Leçon principale** : la simulation Rust permet d'**intuiter** les propriétés de l'algorithme (stabilité, optimalité) sur des instances finies ; la preuve Lean 4 les **garantit** pour tout `n` et tout profil. Les deux se complètent — la simulation pour la dynamique, la preuve pour l'universalité.

**Pour aller plus loin** :

- [Gale & Shapley (1962), College Admissions and the Stability of Marriage](https://www.jstor.org/stable/2312726) — l'article original
- [Knuth (1976), Mariages Stables](https://www-cs-faculty.stanford.edu/~knuth/mariages-stables.html) — introduction canonique en français
- [Roth (2008), Deferred Acceptance Algorithms](https://link.springer.com/article/10.1007/s00182-008-0117-6) — applications modernes (NRMP, 1952-depuis)
- Lake Lean [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) — formalisation sans `sorry`

**Retour au sommaire** : [Social Choice](../README.md) | [GameTheory](../../README.md)

### Références académiques

- Gale, D. & Shapley, L.S. (1962). College Admissions and the Stability of Marriage. **The American Mathematical Monthly** 69(1):9-15.
- Knuth, D.E. (1976). **Mariages Stables et leurs relations avec d'autres problèmes combinatoires**. Les Presses de l'Université de Montréal.
- Roth, A.E. (2008). Deferred Acceptance Algorithms: History, Theory, Practice, and Open Questions. **International Journal of Game Theory** 36(3-4):537-569.
- Gusfield, D. (1989). Three still open problems in the theory of stable marriage. Technical Report CS-89-21, UC Davis.

***

**Navigation** : [<< ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb](ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb) | [ChoixSocial-03-Voting-Methods-Python.ipynb >>](ChoixSocial-03-Voting-Methods-Python.ipynb) | [Index](../README.md) | [twin Python](ChoixSocial-02-Stable-Marriage-Gale-Shapley-Python.ipynb) | [twin C#](ChoixSocial-02-Stable-Marriage-Gale-Shapley-CSharp.ipynb)

> Carnet issu de l'EPIC #26534.